# Figures of the paper

Regenerates the ten data figures of *Recovering Irreducible Three-Body Interactions with Lagrangian Neural Networks*
into `figures/`, with the file names used in the manuscript (Fig. 2, the architecture schematic, is drawn in LaTeX).
The trained models are loaded from `weights/` (rebuilt from plain weights, see `lnn/release_io.py`), datasets from `data/`.

| Paper | File |
|---|---|
| Fig. 1 | `at_familiarity_schematic.pdf` (analytic) |
| Fig. 3 | `sampling_ablation.pdf` |
| Fig. 4 | `fig1_field_ticked.pdf` (slow: ~1–2 min) |
| Fig. 5 | `fig2_radial_raw.pdf` |
| Fig. 6 | `fig2_radial_raw_lj.pdf` |
| Fig. 7 | `fig7_newtonian_control.pdf` |
| Fig. 8 | `fig3_angular_raw.pdf` |
| Fig. 9a, 9b | `fig4a_curved.pdf`, `fig4b_ensemble.pdf` (9b reads the precomputed N = 100 ensemble statistics in `results/`) |
| Fig. 10 | `fig5_special.pdf` |
| Fig. 11 | `fig6_identifiability.pdf` |

Learned-model rollouts run in a separate process (`scripts/rollout_worker.py`, adaptive torchode solver).
The numbers printed by the cells are those quoted in the paper; the last cell lists them.

In [ ]:
# Repository paths (run from the repository root or from notebooks/)
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LNN_DIR, WEIGHTS, DATA = REPO / "lnn", REPO / "weights", REPO / "data"
RESULTS, FIGURES = REPO / "results", REPO / "figures"
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(LNN_DIR))

In [ ]:
import warnings
warnings.filterwarnings('ignore', message=".*timestamp.*")
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import ImageGrid
"""
Setup: same model, same conventions as pure_AT_final_figures.ipynb, but this notebook's own
outputs go to a separate folder so the existing figures/ directory is never touched.
"""
import sys, os
sys.path.insert(0, str(LNN_DIR))

import numpy as np
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt

from physical_systems import three_body, initialize_particle_system
from fixedkinetic_LNN import LNN
torch.serialization.add_safe_globals([LNN])
from file_exports import load_particle
from release_io import load_model_path as load_model

PROJ    = str(WEIGHTS)
RUN_DIR = f"{PROJ}/default_pure_AT_rd15"

OUT_DIR       = str(FIGURES)   # THIS notebook's outputs -- new folder
OUT_DIR_CACHE = str(RESULTS)          # read-only reuse of the cached N=100 ensemble
os.makedirs(OUT_DIR, exist_ok=True)

NU    = 1.0     # AT coupling E0 of this run
R_MIN = 0.6     # training minimum-separation cutoff (pure-AT isolation study)
R_MAX = 5.0     # training maximum pairwise distance

DEVICE, DTYPE = torch.device("cpu"), torch.float32

# ---------------------------------------------------------------------------
# House style -- identical to pure_AT_final_figures.ipynb
# ---------------------------------------------------------------------------
PRE_COL_W = 3.375      # single-column width (in)
PRE_PAGE_W = 7.0       # double-column width (in)

def set_pub_style():
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["cmr10"], "axes.formatter.use_mathtext": True,
        "mathtext.fontset": "cm",
        "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
        "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
        "figure.titlesize": 11, "axes.linewidth": 0.8, "lines.linewidth": 1.4,
        "lines.markersize": 4, "savefig.dpi": 300,
        "pdf.fonttype": 42, "ps.fonttype": 42,
    })

set_pub_style()

C_TRUE  = "#000000"   # analytic: solid black
C_LEARN = "#0072B2"   # learned:  dashed blue
C_ERR   = "#D55E00"   # error / residual: vermillion
CMAP_FIELD = "viridis"
CMAP_ERR   = "magma"
CMAP_SCALE = mpl.cm.plasma

# ---------------------------------------------------------------------------
# Load model + analytic ground-truth system (pure AT)
# ---------------------------------------------------------------------------
model = load_model(f"{RUN_DIR}/model.pt", DEVICE, DTYPE)
model.eval()
particle = load_particle(initialize_particle_system("three_body"),
                         f"{RUN_DIR}/particle.json")
particle.E0 = NU
SCALE = np.array(particle.scale)

print("Loaded model + particle. E0(nu)=", particle.E0, " eps_LJ=", particle.eps_LJ,
      " scale=", particle.scale)

# ---------------------------------------------------------------------------
# Core evaluators (single source of truth for the whole notebook)
# ---------------------------------------------------------------------------
def _learned_potential_raw(q6):
    state = torch.as_tensor(q6, dtype=DTYPE, device=DEVICE).reshape(1, 6)
    with torch.no_grad():
        return -model.lagrangian(state).item()

LEARN_V_OFFSET = 0.0
def learned_potential(q6):
    return _learned_potential_raw(q6) - LEARN_V_OFFSET

def learned_accel(q6):
    q = torch.as_tensor(q6, dtype=DTYPE, device=DEVICE).reshape(6)
    state = torch.cat([q, torch.zeros(6, dtype=DTYPE)]).reshape(1, 12)
    with torch.no_grad():
        return model(state)[0, 6:].cpu().numpy()

def true_potential(q6):
    q = torch.as_tensor(q6, dtype=DTYPE, device=DEVICE).reshape(6)
    return float(particle.potential(q, torch.zeros(6, dtype=DTYPE)))

def true_accel(q6):
    q = torch.as_tensor(q6, dtype=DTYPE, device=DEVICE).reshape(1, 6).clone().requires_grad_(True)
    qd = torch.zeros(1, 6, dtype=DTYPE, requires_grad=True)
    return particle.solve_acceleration(q, qd).detach().cpu().numpy()[0]

def geometry(q6):
    """Pairwise distances and AT angular factor Phi for a 6-vector."""
    xi, yi, xj, yj, xk, yk = q6
    r12 = np.hypot(xi - xj, yi - yj)
    r23 = np.hypot(xj - xk, yj - yk)
    r31 = np.hypot(xk - xi, yk - yi)
    c1 = (r12**2 + r31**2 - r23**2) / (2*r12*r31)
    c2 = (r12**2 + r23**2 - r31**2) / (2*r12*r23)
    c3 = (r31**2 + r23**2 - r12**2) / (2*r31*r23)
    return r12, r23, r31, 1 + 3*c1*c2*c3

def energy_series(path):
    """path:[T,12] np -> total TRUE energy per step (pure AT). Same convention as
    physical_systems.py's three_body.energy() / evaluate.py's particle.energy(x)."""
    q = path[:, :6]; v = path[:, 6:]
    p = q.reshape(-1, 3, 2)
    r12 = np.linalg.norm(p[:, 0]-p[:, 1], axis=1)
    r23 = np.linalg.norm(p[:, 1]-p[:, 2], axis=1)
    r31 = np.linalg.norm(p[:, 2]-p[:, 0], axis=1)
    e = 1e-8
    c1 = (r12**2+r31**2-r23**2)/(2*r12*r31+e)
    c2 = (r12**2+r23**2-r31**2)/(2*r12*r23+e)
    c3 = (r31**2+r23**2-r12**2)/(2*r31*r23+e)
    VAT = NU*(1+3*c1*c2*c3)/((r12*r23*r31)**3)
    T = 0.5*np.sum(v*v, axis=1)
    return T + VAT

# --- calibrate the additive gauge constant of the learned potential ---
def _calibrate_offset(n=400, seed=0):
    rng = np.random.default_rng(seed)
    diffs = []
    while len(diffs) < n:
        pts = rng.uniform(-1.4, 1.4, size=(3, 2))
        r12 = np.linalg.norm(pts[0]-pts[1]); r23 = np.linalg.norm(pts[1]-pts[2])
        r31 = np.linalg.norm(pts[2]-pts[0])
        if min(r12, r23, r31) < R_MIN or max(r12, r23, r31) > R_MAX:
            continue
        q6 = (pts - pts.mean(axis=0)).reshape(-1).astype(np.float32)
        diffs.append(_learned_potential_raw(q6) - true_potential(q6))
    return float(np.median(diffs))

LEARN_V_OFFSET = _calibrate_offset()
print("gauge offset (learned - true) =", LEARN_V_OFFSET)

## Fig. 1: the AT interaction at a glance

In [ ]:
# analytic schematic ##

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

PRE_PAGE_W = 7.0
OUT_DIR = str(FIGURES)

def set_pub_style():
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["cmr10"], "axes.formatter.use_mathtext": True,
        "mathtext.fontset": "cm",
        "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
        "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 8,
        "figure.titlesize": 11, "axes.linewidth": 0.8, "lines.linewidth": 1.4,
        "lines.markersize": 4, "savefig.dpi": 300,
        "pdf.fonttype": 42, "ps.fonttype": 42,
    })

set_pub_style()
CMAP_SCALE = mpl.cm.plasma
NU = 1.0

def at_potential(q6, nu=NU):
    p = q6.reshape(3, 2)
    r12 = np.linalg.norm(p[0]-p[1]); r23 = np.linalg.norm(p[1]-p[2]); r31 = np.linalg.norm(p[2]-p[0])
    c1 = (r12**2 + r31**2 - r23**2) / (2*r12*r31)
    c2 = (r12**2 + r23**2 - r31**2) / (2*r12*r23)
    c3 = (r31**2 + r23**2 - r12**2) / (2*r31*r23)
    Phi = 1 + 3*c1*c2*c3
    return nu * Phi / (r12*r23*r31)**3, Phi

def at_forces(q6, nu=NU, h=1e-6):
    F = np.zeros(6)
    for i in range(6):
        qp = q6.copy(); qp[i] += h
        qm = q6.copy(); qm[i] -= h
        Vp, _ = at_potential(qp, nu); Vm, _ = at_potential(qm, nu)
        F[i] = -(Vp - Vm) / (2*h)
    return F.reshape(3, 2)

def unit_equilateral():
    return np.array([[0, 1/np.sqrt(3)], [-0.5, -1/(2*np.sqrt(3))], [0.5, -1/(2*np.sqrt(3))]], dtype=float)

def unit_right():
    return np.array([[0, 0], [1, 0], [0, 1]], dtype=float)

def unit_near_collinear():
    return np.array([[-1, 0], [1, 0], [0, 0.15]], dtype=float)

SHAPES = [("Equilateral", unit_equilateral), ("Right", unit_right),
          ("Near-collinear", unit_near_collinear)]
SCALES = [0.6, 1.5, 3.0]   # 3 columns

all_data = {}
all_logf = []
for name, base_fn in SHAPES:
    base = base_fn(); base -= base.mean(axis=0)
    for s in SCALES:
        pts = base * s
        q6 = pts.reshape(-1)
        F = at_forces(q6)
        fmag = np.linalg.norm(F, axis=1)
        _, Phi = at_potential(q6)
        all_data[(name, s)] = (pts, F, fmag, Phi)
        all_logf.append(np.log10(fmag + 1e-300))

norm = mpl.colors.Normalize(vmin=np.concatenate(all_logf).min(),
                            vmax=np.concatenate(all_logf).max())

fig, axes = plt.subplots(len(SHAPES), len(SCALES),
                         figsize=(PRE_PAGE_W, PRE_PAGE_W * 0.30 * len(SHAPES)))
labels = ["1", "2", "3"]

for row, (name, _) in enumerate(SHAPES):
    for col, s in enumerate(SCALES):
        ax = axes[row, col]
        pts, F, fmag, Phi = all_data[(name, s)]
        logf = np.log10(fmag + 1e-300)

        tri = np.vstack([pts, pts[0]])
        ax.plot(tri[:, 0], tri[:, 1], color="0.6", lw=1.0, zorder=2)

        arrow_len = 0.5 * s          # was 0.6*s — smaller so arrows fit inside the box
        qscale = max(fmag.max() / arrow_len, 1e-12)
        ax.quiver(pts[:, 0], pts[:, 1], F[:, 0], F[:, 1], logf,
                  cmap=CMAP_SCALE, norm=norm, angles="xy", scale_units="xy",
                  scale=qscale, width=0.02, zorder=3)

        ax.scatter(pts[:, 0], pts[:, 1], s=32, color="k", zorder=4)
        for p, lab in zip(pts, labels):
            ax.annotate(lab, p, textcoords="offset points", xytext=(4, 4), fontsize=6)

        pad = 0.55 * s
        ax.set_xlim(pts[:, 0].min() - pad, pts[:, 0].max() + pad)
        ax.set_ylim(pts[:, 1].min() - pad, pts[:, 1].max() + pad)
        ax.set_aspect("equal", adjustable="datalim")
        ax.grid(True, alpha=0.3, lw=0.4, zorder=0)
        ax.tick_params(length=2)

        if row == 0:
            ax.set_title(f"$s={s:.1f}$", fontsize=9)
        if col == 0:
            ax.set_ylabel(name, fontsize=9)

fig.subplots_adjust(right=0.88, wspace=0.32, hspace=0.15)   # was 0.25, 0.30
cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
sm = mpl.cm.ScalarMappable(norm=norm, cmap=CMAP_SCALE)
fig.colorbar(sm, cax=cax, label=r"$\log_{10}|\mathbf{F}|$")

fig.savefig(f"{OUT_DIR}/at_familiarity_schematic.pdf", bbox_inches="tight")
plt.show()
print("saved to", f"{OUT_DIR}/at_familiarity_schematic.pdf")

## Fig. 3: sampling ablation

In [ ]:
# Sampling ablation: fair, magnitude-normalized functional-form check, r in [0.6, 3.0].
# Self-contained (loads its own models).
import sys, numpy as np, torch
sys.path.insert(0, str(LNN_DIR))
import matplotlib as mpl, matplotlib.pyplot as plt
from physical_systems import three_body
from fixedkinetic_LNN import LNN
torch.serialization.add_safe_globals([LNN])
from release_io import load_model_path as load_model

_PROJ = str(WEIGHTS)
_OUT_DIR = str(FIGURES)
_DEV, _DT = torch.device("cpu"), torch.float32
try:
    set_pub_style()
except NameError:
    mpl.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm", "font.size": 9,
                         "savefig.dpi": 300, "pdf.fonttype": 42})
_R_DENSE = 1.5
_part_sa = three_body(E0=1.0, eps_LJ=0.0)

def _learned_acc_sa(m, q6):
    s = torch.cat([torch.tensor(q6, dtype=_DT), torch.zeros(6)]).reshape(1, 12)
    with torch.no_grad():
        return m(s)[0, 6:].numpy()

def _true_acc_sa(q6):
    q = torch.tensor(q6, dtype=_DT).reshape(1, 6).clone().requires_grad_(True)
    qd = torch.zeros(1, 6, requires_grad=True)
    return _part_sa.solve_acceleration(q, qd).detach().numpy()[0]

def _equilateral_sa(r):
    p = np.array([[0, 0], [r, 0], [0.5*r, np.sqrt(3)/2*r]]); p = p - p.mean(0)
    return p.reshape(-1).astype(np.float32)

_models_sa = {
    "log-uniform":          load_model(f"{_PROJ}/default_pure_AT_loguniform/model.pt", _DEV, _DT).eval(),
    "dense/sparse (90/10)": load_model(f"{_PROJ}/default_pure_AT_rd15/model.pt",        _DEV, _DT).eval(),
}
_rvals_sa = np.linspace(0.6, 3.0, 120)
_err_sa = {n: np.array([np.linalg.norm(_learned_acc_sa(m, _equilateral_sa(r)) - _true_acc_sa(_equilateral_sa(r)))
                        / (np.linalg.norm(_true_acc_sa(_equilateral_sa(r))) + 1e-12)
                        for r in _rvals_sa])
           for n, m in _models_sa.items()}

print("relative acceleration error on the equilateral sweep:")
for n in _models_sa:
    print(f"  {n:22s} r=0.6:{_err_sa[n][0]:.3f}  "
          f"r=1.0:{_err_sa[n][np.argmin(abs(_rvals_sa-1.0))]:.3f}  "
          f"r=1.5:{_err_sa[n][np.argmin(abs(_rvals_sa-1.5))]:.3f}  "
          f"r=2.0:{_err_sa[n][np.argmin(abs(_rvals_sa-2.0))]:.3f}  "
          f"r=2.5:{_err_sa[n][np.argmin(abs(_rvals_sa-2.5))]:.3f}  "
          f"r=3.0:{_err_sa[n][np.argmin(abs(_rvals_sa-3.0))]:.3f}")

fig, ax = plt.subplots(figsize=(3.5, 2.7))
for n, c in zip(_models_sa, ["#D55E00", "#0072B2"]):
    ax.plot(_rvals_sa, _err_sa[n], color=c, lw=1.7, label=n)
ax.axvline(_R_DENSE, color="0.6", lw=0.8, ls="--")
ax.text(_R_DENSE+0.03, ax.get_ylim()[1]*0.6, r"$r_{\rm dense}$", fontsize=7, color="0.4")
ax.set_yscale("log"); ax.set_xlabel(r"equilateral side $r$")
ax.set_ylabel("relative acceleration error")
# (no ax.set_title -- single-panel figure, caption carries it)
ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{_OUT_DIR}/sampling_ablation.pdf", bbox_inches="tight")
print("sampling ablation done")

## Fig. 4: potential landscape and force field (slow: ~1–2 min)

In [ ]:
# FIGURE 1 variant: same grid/config as above, but with real position ticks on the axes,
# and an explicit report of how much of the frame is in the pink/sparsely-sampled periphery.
def fig1_field_ticked(N=161, sep=0.6, span=1.5, r_dense=1.5,
                       v_clip=98, f_lo=8, f_hi=99, ticks=(-1, 0, 1), save=True):
    p1 = np.array([-sep, 0.0]); p2 = np.array([sep, 0.0])
    xs = np.linspace(-span, span, N); ys = np.linspace(-span, span, N)
    X, Y = np.meshgrid(xs, ys)
    trueV = np.full_like(X, np.nan); predV = np.full_like(X, np.nan)
    Fxt = np.full_like(X, np.nan); Fyt = np.full_like(X, np.nan)
    Fxp = np.full_like(X, np.nan); Fyp = np.full_like(X, np.nan)
    Rmin_grid = np.full_like(X, np.nan)

    for i in range(N):
        for j in range(N):
            p3 = np.array([X[i, j], Y[i, j]])
            r13 = np.linalg.norm(p3 - p1); r23 = np.linalg.norm(p3 - p2)
            if min(r13, r23) < R_MIN:
                continue
            Rmin_grid[i, j] = min(r13, r23)
            pts = np.vstack([p1, p2, p3]); pts = pts - pts.mean(axis=0)
            q6 = pts.reshape(-1).astype(np.float32)
            trueV[i, j] = true_potential(q6); predV[i, j] = learned_potential(q6)
            at = true_accel(q6); ap = learned_accel(q6)
            Fxt[i, j], Fyt[i, j] = at[4], at[5]
            Fxp[i, j], Fyp[i, j] = ap[4], ap[5]

    Verr = np.abs(predV - trueV)
    Fmt = np.hypot(Fxt, Fyt); Fmp = np.hypot(Fxp, Fyp)
    Ferr = np.hypot(Fxt - Fxp, Fyt - Fyp)

    maskV = np.isfinite(trueV)
    valid = np.isfinite(Rmin_grid)
    tc = trueV - np.nanmean(trueV); pc = predV - np.nanmean(predV)
    rel_V = np.sum(np.abs(pc[maskV] - tc[maskV])) / np.sum(np.abs(tc[maskV]))
    rel_F = np.sum(Ferr[np.isfinite(Fmt)]) / np.sum(Fmt[np.isfinite(Fmt)])
    print(f"(sep={sep}, span={span})  aggregate rel V = {100*rel_V:.2f}%   rel F = {100*rel_F:.2f}%")

    periphery = Rmin_grid >= r_dense
    frac_grid  = periphery.sum() / periphery.size
    frac_valid = (periphery & valid).sum() / valid.sum()
    print(f"pink/periphery (r_min >= r_dense={r_dense}): {100*frac_grid:.1f}% of the full grid, "
          f"{100*frac_valid:.1f}% of the plotted (unmasked) area | "
          f"max r_min in frame = {np.nanmax(Rmin_grid):.3f}")

    pink = np.zeros(X.shape + (4,))
    pink[..., 0] = 1.0; pink[..., 1] = 0.85; pink[..., 2] = 0.88
    pink[..., 3] = np.where(periphery, 0.30, 0.0)

    allV = np.concatenate([tc[maskV], pc[maskV]])
    vnorm = mcolors.Normalize(np.nanpercentile(allV, 100 - v_clip), np.nanpercentile(allV, v_clip))
    enorm = mcolors.Normalize(0, np.nanpercentile(Verr[np.isfinite(Verr)], v_clip))
    mags = np.concatenate([Fmt[np.isfinite(Fmt)], Fmp[np.isfinite(Fmp)]])
    fnorm = mcolors.LogNorm(vmin=max(np.nanpercentile(mags, f_lo), 1e-4), vmax=np.nanpercentile(mags, f_hi))
    frnorm = mcolors.LogNorm(vmin=max(np.nanpercentile(Ferr[np.isfinite(Ferr)], f_lo), 1e-4),
                             vmax=np.nanpercentile(Ferr[np.isfinite(Ferr)], f_hi))

    cmap_V = plt.cm.viridis.copy();  cmap_V.set_bad("0.80")
    cmap_E = plt.cm.magma.copy();    cmap_E.set_bad("0.80")

    fig = plt.figure(figsize=(PRE_PAGE_W, PRE_PAGE_W*0.68))
    grid = ImageGrid(fig, 111, nrows_ncols=(2, 3), axes_pad=(0.55, 0.50),
                     share_all=True, aspect=True, label_mode="L",
                     cbar_mode="each", cbar_size="5%", cbar_pad=0.05)
    ext = [-span, span, -span, span]
    titles = ["Analytic $V$", "Learned $V$", "$|V_\\mathrm{learn}-V_\\mathrm{true}|$",
              "Analytic force field", "Learned force field", "Force residual"]

    def show(g, arr, cmap, norm, label):
        im = g.imshow(arr, extent=ext, origin="lower", cmap=cmap, norm=norm, interpolation="nearest")
        cb = g.cax.colorbar(im); cb.set_label(label, fontsize=7); g.cax.tick_params(labelsize=6)

    show(grid[0], tc,   cmap_V, vnorm, r"$V-\langle V\rangle$")
    show(grid[1], pc,   cmap_V, vnorm, r"$V-\langle V\rangle$")
    show(grid[2], Verr, cmap_E, enorm, r"$|\Delta V|$")

    streams = [(Fxt, Fyt, Fmt, fnorm), (Fxp, Fyp, Fmp, fnorm), (Fxt-Fxp, Fyt-Fyp, Ferr, frnorm)]
    clabels = [r"$|\mathbf{F}|$", r"$|\mathbf{F}|$", r"$|\Delta\mathbf{F}|$"]
    for k, (fx, fy, col, nrm) in enumerate(streams):
        g = grid[3 + k]
        g.imshow(pink, extent=ext, origin="lower", zorder=0)
        strm = g.streamplot(X, Y, fx, fy, color=col, density=1.5, cmap=CMAP_SCALE,
                            norm=nrm, linewidth=1.0, arrowsize=0.9, zorder=2)
        cb = g.cax.colorbar(strm.lines); cb.set_label(clabels[k], fontsize=7); g.cax.tick_params(labelsize=6)

    for k, g in enumerate(grid):
        g.scatter([p1[0], p2[0]], [0, 0], s=45, c="white", edgecolors="black", zorder=10)
        g.set_xticks(ticks); g.set_yticks(ticks)
        g.tick_params(labelsize=7)
        g.set_title(titles[k], fontsize=9)
        g.set_xlim(-span, span); g.set_ylim(-span, span)
    # bottom row gets the x label, left column gets the y label (ImageGrid's label_mode="L"
    # already hides the redundant inner tick-number labels; this adds the axis names on top)
    for k in (3, 4, 5):
        grid[k].set_xlabel("$x$", fontsize=9)
    for k in (0, 3):
        grid[k].set_ylabel("$y$", fontsize=9)

    # (no fig.suptitle -- caption carries the config/summary text in the paper)
    if save:
        fig.savefig(f"{OUT_DIR}/fig1_field_ticked.pdf", bbox_inches="tight")
    print("fig1 (ticked) done")
    return fig

fig1_field_ticked(sep=0.6, span=1.5)

## Fig. 5: radial dependence (pure AT)

In [ ]:
# ===========================================================================
# FIGURE 2 : radial dependence, 6 configurations, potential AND acceleration
# ===========================================================================
def _center(pts):
    pts = pts - pts.mean(axis=0)
    return pts.reshape(-1).astype(np.float32)

def build_equilateral(r): return _center(np.array([[0,0],[r,0],[0.5*r, np.sqrt(3)/2*r]]))
def build_right(r):       return _center(np.array([[0,0],[r,0],[0,r]]))
def build_line(r):        return _center(np.array([[-r,0],[0,0],[r,0]]))
def build_far(r):         return _center(np.array([[0,0],[r,0],[0,3*r]]))
def build_isosceles(r, sr=1.25):
    s = sr*r
    h = np.sqrt(s**2 - (r/2)**2)
    return _center(np.array([[0,0],[r,0],[r/2,h]]))
def build_scalene(r):     return _center(np.array([[0,0],[1.2*r,0],[0.35*r,1.05*r]]))

CONFIGS = {
    "Equilateral": build_equilateral,
    "Right": build_right,
    "Line": build_line,
    "Far particle": build_far,
    "Isosceles": build_isosceles,
    "Scalene": build_scalene,
}

def fig2_radial(rvals=None, save=True):
    if rvals is None:
        rvals = np.linspace(R_MIN, 1.6, 90)

    fig, axes = plt.subplots(4, 3, figsize=(PRE_PAGE_W, PRE_PAGE_W*1.05))

    for c, (name, builder) in enumerate(CONFIGS.items()):
        col = c % 3
        block = c // 3
        rowV = 0 if block == 0 else 1
        rowA = 2 if block == 0 else 3

        Vt, Vp, At, Ap = [], [], [], []

        for r in rvals:
            q6 = builder(r)
            Vt.append(true_potential(q6))
            Vp.append(learned_potential(q6))
            At.append(np.linalg.norm(true_accel(q6)))
            Ap.append(np.linalg.norm(learned_accel(q6)))

        Vt = np.asarray(Vt)
        Vp = np.asarray(Vp)
        At = np.asarray(At)
        Ap = np.asarray(Ap)

        # Center potentials (energy offset is arbitrary)
        Vt -= Vt.mean()
        Vp -= Vp.mean()

        # Reference power laws (also centered for comparison)
        refV = Vt[0] * (rvals / rvals[0])**(-9)
        refV -= refV.mean()
        refA = At[0] * (rvals / rvals[0])**(-10)

        axv = axes[rowV, col]
        axv.plot(rvals, Vt, color=C_TRUE, label="Analytic")
        axv.plot(rvals, Vp, color=C_LEARN, ls="--", label="Learned")
        axv.plot(rvals, refV, color="0.6", lw=0.8, ls=":", label=r"$r^{-9}$")
        axv.set_title(name, fontsize=9)
        axv.grid(alpha=0.3)
        if col == 0:
            axv.set_ylabel(r"$V-\langle V\rangle$")

        axa = axes[rowA, col]
        axa.plot(rvals, At, color=C_TRUE)
        axa.plot(rvals, Ap, color=C_LEARN, ls="--")
        axa.plot(rvals, refA, color="0.6", lw=0.8, ls=":")
        axa.grid(alpha=0.3)
        axa.set_xlabel(r"$r$")
        if col == 0:
            axa.set_ylabel(r"$|\mathbf{a}|$")

    axes[0, 0].legend(loc="lower left", fontsize=6)

    fig.tight_layout()

    if save:
        fig.savefig(f"{OUT_DIR}/fig2_radial_raw.pdf", bbox_inches="tight")

    print("fig2 done")
    return fig


# ===========================================================================

fig2_radial()

## Fig. 8: angular dependence

In [ ]:
# ===========================================================================
# FIGURE 3 : angular dependence (fixed length scale)
# ===========================================================================
def fig3_angular(scale=1.0, n=180, amin=12, amax=168, save=True):

    theta_min = 2*np.arcsin(0.3) + 1e-3
    theta_max = np.pi - 0.2

    angles = np.linspace(theta_min, theta_max, n)
    angles_deg = np.degrees(angles)

    Vt, Vp = [], []
    At, Ap = [], []

    p1 = np.array([0.0, 0.0])
    p2 = np.array([scale, 0.0])

    for th in angles:

        p3 = np.array([
            scale * np.cos(th),
            scale * np.sin(th)
        ])

        pts = np.vstack([p1, p2, p3])
        pts -= pts.mean(axis=0)
        q6 = pts.reshape(-1).astype(np.float32)

        # potentials
        vt = true_potential(q6)
        vp = learned_potential(q6)

        # accelerations
        at = np.linalg.norm(true_accel(q6))
        ap = np.linalg.norm(learned_accel(q6))

        Vt.append(vt)
        Vp.append(vp)
        At.append(at)
        Ap.append(ap)

    Vt = np.asarray(Vt)
    Vp = np.asarray(Vp)
    At = np.asarray(At)
    Ap = np.asarray(Ap)

    # -------------------------
    # Center potentials
    # -------------------------
    Vt -= Vt.mean()
    Vp -= Vp.mean()

    # -------------------------
    # Relative mean errors
    # -------------------------
    eps = 1e-12

    relV = np.mean(np.abs(Vp - Vt) / np.maximum(np.abs(Vt), eps))
    relA = np.mean(np.abs(Ap - At) / np.maximum(np.abs(At), eps))

    # -------------------------
    # Plot
    # -------------------------
    fig, axes = plt.subplots(
        1, 2,
        figsize=(PRE_PAGE_W, PRE_PAGE_W * 0.42)
    )

    # Potential
    ax = axes[0]
    ax.axhline(0, color="0.6", lw=0.8, ls=":")
    ax.plot(angles_deg, Vt, color=C_TRUE, label="Analytic")
    ax.plot(angles_deg, Vp, color=C_LEARN, ls="--", label="Learned")
    ax.set_xlabel("included angle (deg)")
    ax.set_ylabel(r"$V-\langle V\rangle$")
    ax.set_title("Potential")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)

    # Acceleration
    ax = axes[1]
    ax.plot(angles_deg, At, color=C_TRUE)
    ax.plot(angles_deg, Ap, color=C_LEARN, ls="--")
    ax.set_xlabel("included angle (deg)")
    ax.set_ylabel(r"$|\mathbf{a}|$")
    ax.set_title("Acceleration magnitude")
    ax.grid(alpha=0.3)


    fig.tight_layout()

    if save:
        fig.savefig(f"{OUT_DIR}/fig3_angular_raw.pdf", bbox_inches="tight")

    print(
        f"fig3 done | "
        f"Potential mean relative error = {100*relV:.3f}% | "
        f"Acceleration mean relative error = {100*relA:.3f}%"
    )

    return fig


# ===========================================================================

fig3_angular()

## Fig. 9: trajectory rollouts (a: three cases, b: cached N=100 ensemble, read-only)

In [ ]:
# FIGURE 4: trajectory rollouts, self-contained (own model/particle load).
import sys, numpy as np, torch
sys.path.insert(0, str(LNN_DIR))
import matplotlib as mpl, matplotlib.pyplot as plt
from physical_systems import three_body, initialize_particle_system
from fixedkinetic_LNN import LNN
torch.serialization.add_safe_globals([LNN])
from file_exports import load_particle
from release_io import load_model_path as load_model

_PROJ = str(WEIGHTS)
_RUN_DIR = f"{_PROJ}/default_pure_AT_rd15"
_OUT_DIR = str(FIGURES)          # THIS notebook's outputs
_OUT_DIR_CACHE = str(RESULTS)           # read-only: cached N=100 ensemble
_NU, _R_MIN, _R_MAX = 1.0, 0.6, 5.0
_DEVICE, _DTYPE = torch.device("cpu"), torch.float32
_PAGE_W = 7.0

mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["cmr10"], "axes.formatter.use_mathtext": True,
    "mathtext.fontset": "cm", "font.size": 9, "axes.titlesize": 8.5, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "figure.titlesize": 11, "axes.linewidth": 0.8, "lines.linewidth": 1.4,
    "savefig.dpi": 300, "pdf.fonttype": 42, "ps.fonttype": 42,
})
_C_TRUE = "#000000"
_pcol = ["#0072B2", "#009E73", "#CC79A7"]

_model4 = load_model(f"{_RUN_DIR}/model.pt", _DEVICE, _DTYPE).eval()
_particle4 = load_particle(initialize_particle_system("three_body"), f"{_RUN_DIR}/particle.json")
_particle4.E0 = _NU

import subprocess, tempfile, os
_LNN_ENV_PYTHON = sys.executable
_ROLLOUT_WORKER = str(REPO / "scripts" / "rollout_worker.py")

def _rollout_analytic4(x0, t):
    dt = float(t[1]-t[0]); x = x0.clone(); out=[x]
    def deriv(x):
        q = x[:, :6].clone().detach().requires_grad_(True)
        qd = x[:, 6:].clone().detach().requires_grad_(True)
        a = _particle4.solve_acceleration(q, qd)
        return torch.cat([qd.detach(), a.detach()], dim=1)
    for _ in range(len(t)-1):
        k1=deriv(x); k2=deriv(x+0.5*dt*k1); k3=deriv(x+0.5*dt*k2); k4=deriv(x+dt*k3)
        x=(x+(dt/6.0)*(k1+2*k2+2*k3+k4)).detach(); out.append(x)
    return torch.stack(out, dim=1)

def _rollout_learned4(x0, t, chunk=4):
    with tempfile.TemporaryDirectory() as d:
        in_path, out_path = f"{d}/in.npz", f"{d}/out.npz"
        np.savez(in_path, run_dir=_RUN_DIR, x0=x0.detach().cpu().numpy().astype(np.float32),
                 t=t.detach().cpu().numpy().astype(np.float32), chunk=chunk)
        env = os.environ.copy()
        for v in ("PYTHONHOME","PYTHONPATH","CONDA_PREFIX","CONDA_DEFAULT_ENV","CONDA_PYTHON_EXE","CONDA_SHLVL","VIRTUAL_ENV"):
            env.pop(v, None)
        r = subprocess.run([_LNN_ENV_PYTHON, _ROLLOUT_WORKER, in_path, out_path], env=env, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout); print(r.stderr); raise RuntimeError("rollout worker failed")
        return np.load(out_path)["traj"]

def _rotating_state4(pts, omega, extra_v=None):
    pts = pts - pts.mean(axis=0)
    v = omega * np.stack([-pts[:, 1], pts[:, 0]], axis=1)
    if extra_v is not None: v = v + extra_v
    v = v - v.mean(axis=0)
    return np.concatenate([pts.reshape(-1), v.reshape(-1)]).astype(np.float32)

def fig4_panel_A(save=True):
    fig, ax = plt.subplots(1, 3, figsize=(_PAGE_W, _PAGE_W*0.36))

    T1, dt1 = 5.0, 0.005
    t1 = torch.arange(0.0, T1+0.5*dt1, dt1, dtype=_DTYPE)
    x0_1 = torch.tensor(_rotating_state4(np.array([[-1.,0.],[1.,0.],[0.,0.18]])*1.35, omega=0.28),
                        dtype=_DTYPE).unsqueeze(0)
    tru1 = _rollout_analytic4(x0_1, t1).numpy()[0]
    prd1 = _rollout_learned4(x0_1, t1)[0]
    a = ax[0]
    L1 = 1.1*np.abs(tru1[:, :6]).max()
    for b in range(3):
        a.plot(tru1[:,2*b], tru1[:,2*b+1], color=_pcol[b], lw=1.3)
        a.plot(prd1[:,2*b], prd1[:,2*b+1], color=_pcol[b], lw=1.1, ls="--", alpha=0.8)
        a.scatter([tru1[0,2*b]],[tru1[0,2*b+1]], color=_pcol[b], s=16, zorder=5)
    a.set_xlim(-L1,L1); a.set_ylim(-L1,L1); a.set_aspect("equal")
    a.set_title("(i) attractive, unbound: curves\nthen escapes ($t\\in[0,%.0f]$)"%T1, fontsize=8)
    a.grid(alpha=0.3, lw=0.5); a.tick_params(labelsize=6)
    a.set_xlabel("$x$", fontsize=7); a.set_ylabel("$y$", fontsize=7)

    d = np.load(f"{RESULTS}/fig9b_ensemble_summary.npz")
    cut_big, t_big = d["cut_big"].astype(int), d["t_big"]   # the release ships this one trajectory only
    k2 = 34; n2 = int(cut_big[k2])
    assert int(d["case_ii_index"]) == k2
    tru2 = d["tru_case_ii"][:n2]; prd2 = d["prd_case_ii"][:n2]
    a = ax[1]
    L2 = 1.1*np.abs(tru2[:, :6]).max()
    for b in range(3):
        a.plot(tru2[:,2*b], tru2[:,2*b+1], color=_pcol[b], lw=1.3)
        a.plot(prd2[:,2*b], prd2[:,2*b+1], color=_pcol[b], lw=1.1, ls="--", alpha=0.8)
        a.scatter([tru2[0,2*b]],[tru2[0,2*b+1]], color=_pcol[b], s=16, zorder=5)
    a.set_xlim(-L2,L2); a.set_ylim(-L2,L2); a.set_aspect("equal")
    a.set_title("(ii) attractive, bound: genuine\ninspiral ($t\\in[0,%.2f]$)"%t_big[n2-1], fontsize=8)
    a.grid(alpha=0.3, lw=0.5); a.tick_params(labelsize=6)
    a.set_xlabel("$x$", fontsize=7); a.set_ylabel("$y$", fontsize=7)

    T3, dt3 = 0.45, 0.002
    t3 = torch.arange(0.0, T3+0.5*dt3, dt3, dtype=_DTYPE)
    base_r = np.array([[0,1/np.sqrt(3)],[-0.5,-1/(2*np.sqrt(3))],[0.5,-1/(2*np.sqrt(3))]])
    x0_3 = torch.tensor(_rotating_state4(base_r*0.75, omega=0.35), dtype=_DTYPE).unsqueeze(0)
    tru3 = _rollout_analytic4(x0_3, t3).numpy()[0]
    prd3 = _rollout_learned4(x0_3, t3)[0]
    a = ax[2]
    L3 = 1.1*np.abs(tru3[:, :6]).max()
    for b in range(3):
        a.plot(tru3[:,2*b], tru3[:,2*b+1], color=_pcol[b], lw=1.3)
        a.plot(prd3[:,2*b], prd3[:,2*b+1], color=_pcol[b], lw=1.1, ls="--", alpha=0.8)
        a.scatter([tru3[0,2*b]],[tru3[0,2*b+1]], color=_pcol[b], s=16, zorder=5)
    a.set_xlim(-L3,L3); a.set_ylim(-L3,L3); a.set_aspect("equal")
    a.set_title("(iii) repulsive: brief bend,\nthen straightens ($t\\in[0,%.2f]$)"%T3, fontsize=8)
    a.grid(alpha=0.3, lw=0.5); a.tick_params(labelsize=6)
    a.set_xlabel("$x$", fontsize=7); a.set_ylabel("$y$", fontsize=7)

    ax[0].plot([], [], color="0.3", ls="-", label="Analytic")
    ax[0].plot([], [], color="0.3", ls="--", label="Learned")
    ax[0].legend(loc="upper right", fontsize=6.5)
    # (no fig.suptitle)
    fig.tight_layout()
    if save:
        fig.savefig(f"{_OUT_DIR}/fig4a_curved.pdf", bbox_inches="tight")
    print("panel A done")
    return fig

def fig4_panel_B(save=True):
    d = np.load(f"{RESULTS}/fig9b_ensemble_summary.npz")
    perr, edrift, cdrift = d["perr_big"], d["edrift_big"], d["cdrift_big"]
    tt, n_alive = d["t_big"], d["n_alive"]
    N_BIG, T_BIG = int(d["N_BIG"]), float(d["T_BIG"])

    fig, ax = plt.subplots(2, 2, figsize=(_PAGE_W*0.92, _PAGE_W*0.72))
    ax = ax.ravel()

    a = ax[0]
    lo = np.nanpercentile(perr,10,axis=0); hi = np.nanpercentile(perr,90,axis=0)
    a.fill_between(tt, lo, hi, color="0.8", alpha=0.7)
    a.plot(tt, np.nanmedian(perr,axis=0), color=_C_TRUE, lw=1.6)
    a.set_title("COM-centered position error", fontsize=9)
    a.set_xlabel("$t$"); a.set_ylabel("mean particle error"); a.grid(alpha=0.3)

    a = ax[1]
    for e in edrift: a.plot(tt, e, color="0.8", lw=0.4)
    a.plot(tt, np.nanmedian(edrift,axis=0), color=_C_TRUE, lw=1.6)
    a.set_yscale("log"); a.set_title("Relative energy drift (learned)", fontsize=9)
    a.set_xlabel("$t$"); a.set_ylabel(r"$|E-E_0|/|E_0|$"); a.grid(alpha=0.3, which="both")

    a = ax[2]
    for c in cdrift: a.plot(tt, c, color="0.8", lw=0.4)
    a.plot(tt, np.nanmedian(cdrift,axis=0), color=_C_TRUE, lw=1.6)
    a.set_title("Center-of-mass drift", fontsize=9)
    a.set_xlabel("$t$"); a.set_ylabel(r"$|\mathrm{COM}_\mathrm{true}-\mathrm{COM}_\mathrm{learn}|$", labelpad=2)
    a.grid(alpha=0.3)

    a = ax[3]
    a.step(tt, n_alive, color="#D55E00", lw=1.6, where="post")
    a.set_title("Trajectories still valid", fontsize=9)
    a.set_xlabel("$t$"); a.set_ylabel("$N$ contributing")
    a.set_ylim(0, N_BIG*1.05); a.grid(alpha=0.3)

    for a in ax:
        a.tick_params(labelsize=7.5)

    # (no fig.suptitle)
    fig.tight_layout(h_pad=2.6, w_pad=2.4)
    if save:
        fig.savefig(f"{_OUT_DIR}/fig4b_ensemble.pdf", bbox_inches="tight")
    print("panel B done")
    return fig

fig4_panel_A()
fig4_panel_B()

## Fig. 10: the four canonical configurations

Panel names follow the caption's order (equilateral, right, collinear, far-removed). The printed initial side lengths
confirm the order.

In [ ]:
# FIGURE 5: four canonical Axilrod-Teller configurations.
def fig5_special(T=5.0, dt=2e-3, save=True):
    import glob
    x0 = torch.tensor(np.load(f"{DATA}/special_initial_states.npy"), dtype=DTYPE)
    t = torch.arange(0.0, T + 0.5*dt, dt, dtype=DTYPE)

    def _deriv(x):
        q  = x[:, :6].clone().detach().requires_grad_(True)
        qd = x[:, 6:].clone().detach().requires_grad_(True)
        a  = particle.solve_acceleration(q, qd)
        return torch.cat([qd.detach(), a.detach()], dim=1)

    def _rollout_analytic(x0, t):
        dt_ = float(t[1] - t[0]); x = x0.clone(); out = [x]
        for _ in range(len(t) - 1):
            k1 = _deriv(x); k2 = _deriv(x + 0.5*dt_*k1)
            k3 = _deriv(x + 0.5*dt_*k2); k4 = _deriv(x + dt_*k3)
            x = (x + (dt_/6.0)*(k1 + 2*k2 + 2*k3 + k4)).detach()
            out.append(x)
        return torch.stack(out, dim=1)

    def _rollout_learned(x0, t, chunk=4):
        import subprocess, tempfile, os
        LNN_ENV_PYTHON = sys.executable
        ROLLOUT_WORKER = str(REPO / "scripts" / "rollout_worker.py")
        with tempfile.TemporaryDirectory() as d:
            in_path, out_path = f"{d}/in.npz", f"{d}/out.npz"
            np.savez(in_path, run_dir=RUN_DIR, x0=x0.detach().cpu().numpy().astype(np.float32),
                     t=t.detach().cpu().numpy().astype(np.float32), chunk=chunk)
            env = os.environ.copy()
            for v in ("PYTHONHOME","PYTHONPATH","CONDA_PREFIX","CONDA_DEFAULT_ENV","CONDA_PYTHON_EXE","CONDA_SHLVL","VIRTUAL_ENV"):
                env.pop(v, None)
            r = subprocess.run([LNN_ENV_PYTHON, ROLLOUT_WORKER, in_path, out_path], env=env, capture_output=True, text=True)
            if r.returncode != 0:
                print(r.stdout); print(r.stderr); raise RuntimeError("rollout worker failed")
            return np.load(out_path)["traj"]

    tru = _rollout_analytic(x0, t).cpu().numpy()
    prd = _rollout_learned(x0, t)

    # Stop each trajectory where it leaves the trained range (a side < R_MIN), e.g. the collinear collapse,
    # instead of integrating through the singularity. (The far-removed case starts with a side > R_MAX by
    # design, so only the lower bound is applied.)
    def _cut_at_collapse(tr):
        tr = np.array(tr, dtype=float, copy=True)
        p = tr[..., :6].reshape(*tr.shape[:-1], 3, 2)
        dmin = np.minimum.reduce([np.linalg.norm(p[..., a, :] - p[..., b, :], axis=-1) for a, b in ((0, 1), (1, 2), (2, 0))])
        for k in range(len(tr)):
            bad = np.where(dmin[k] < R_MIN)[0]
            if len(bad):
                tr[k, bad[0]:] = np.nan
                print(f"  case {k+1}: leaves the trained range at t = {float(t[bad[0]]):.3f}")
        return tr
    for k in range(len(x0)):
        print(f"  case {k+1}: initial sides", np.round(np.sort(np.linalg.norm(
            x0[k, :6].numpy().reshape(3, 2)[[0, 1, 2]] - x0[k, :6].numpy().reshape(3, 2)[[1, 2, 0]], axis=1)), 3))
    print("analytic:"); tru = _cut_at_collapse(tru)
    print("learned:");  prd = _cut_at_collapse(prd)

    fig, ax = plt.subplots(1, 4, figsize=(PRE_PAGE_W, PRE_PAGE_W*0.36))
    pcol = ["#0072B2", "#009E73", "#CC79A7"]
    for i in range(4):
        a = ax[i]
        for b in range(3):
            a.plot(tru[i, :, 2*b], tru[i, :, 2*b+1], color=pcol[b], lw=1.2)
            a.plot(prd[i, :, 2*b], prd[i, :, 2*b+1], color=pcol[b], lw=1.0, ls="--", alpha=0.8)
            a.scatter([tru[i, 0, 2*b]], [tru[i, 0, 2*b+1]], color=pcol[b], s=14, zorder=5)
        a.set_aspect("equal", adjustable="datalim")
        a.grid(alpha=0.3, lw=0.5); a.tick_params(labelsize=6)
        a.set_xlabel("$x$", fontsize=7); a.set_ylabel("$y$", fontsize=7)
        a.set_title(["Equilateral", "Right", "Collinear", "Far-removed atom"][i], fontsize=9)
    ax[0].plot([], [], color="0.3", ls="-", label="Analytic")
    ax[0].plot([], [], color="0.3", ls="--", label="Learned")
    ax[0].legend(loc="upper right", fontsize=6)
    # (no fig.suptitle -- clean notebook's no-title convention, caption carries it)
    # rect matches pure_AT_final_figures.ipynb's fig.tight_layout(rect=[0,0,1,0.92]),
    # which reserved the top 8% for its suptitle -- keeping the same rect here (with
    # no title drawn into it) makes the axes occupy the same physical fraction of the
    # canvas, so panel size/shape matches that notebook's fig5 exactly.
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    if save:
        fig.savefig(f"{OUT_DIR}/fig5_special.pdf", bbox_inches="tight")
    print("fig5 done")
    return fig

fig5_special()

## Fig. 11: identifiability on the LJ background

In [ ]:
# FIGURE 6: identifiability of the three-body term on a pairwise (LJ) background.
import sys, os
sys.path.insert(0, str(LNN_DIR))
import numpy as np, torch
import matplotlib as mpl, matplotlib.pyplot as plt
from scipy.stats import binned_statistic
from physical_systems import three_body, initialize_particle_system
from fixedkinetic_LNN import LNN
torch.serialization.add_safe_globals([LNN])
from release_io import load_model_path as load_model

_PROJ = str(WEIGHTS)
_OUT_DIR = str(FIGURES)
_DEV, _DT = torch.device("cpu"), torch.float32
try:
    set_pub_style()
except NameError:
    mpl.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm",
                         "font.size": 9, "savefig.dpi": 300, "pdf.fonttype": 42})
_PAGE_W = 7.0
_C_FLOOR, _C_PHYS, _C_AMP = "#000000", "#0072B2", "#D55E00"

_M_LJ   = f"{_PROJ}/argonlike-liquid_pure_LJ"
_M_AMP  = f"{_PROJ}/argonlike-amplified_version"
_M_PHYS = f"{_PROJ}/argonlike-liquid_second_version"
_NU_PHYS, _NU_AMP = 0.1, 1.0

_model_lj   = load_model(f"{_M_LJ}/model.pt",   _DEV, _DT).eval()
_model_amp  = load_model(f"{_M_AMP}/model.pt",  _DEV, _DT).eval()
_model_phys = load_model(f"{_M_PHYS}/model.pt", _DEV, _DT).eval()

_p_lj   = three_body(E0=0.0, eps_LJ=1.0)
_p_at1  = three_body(E0=1.0, eps_LJ=0.0)
_p_amp  = three_body(E0=_NU_AMP,  eps_LJ=1.0)
_p_phys = three_body(E0=_NU_PHYS, eps_LJ=1.0)

_val = torch.load(f"{DATA}/argonlike_liquid_24k_maxr_3_minr09_validation_dataset.pt",
                  weights_only=False)
_states6 = (_val.tensors[0] if hasattr(_val, "tensors") else _val[0]).to(_DT)
_rng6 = np.random.default_rng(0)
_idx6 = _rng6.choice(len(_states6), size=min(2000, len(_states6)), replace=False)
_states6 = _states6[_idx6]

def _model_accel6(model, s):
    out = []
    for i in range(0, len(s), 200):
        with torch.no_grad():
            out.append(model(s[i:i+200])[:, 6:].cpu().numpy())
    return np.concatenate(out, 0)

def _analytic_accel6(p, s):
    q = s[:, :6].clone().requires_grad_(True); qd = s[:, 6:].clone().requires_grad_(True)
    return p.solve_acceleration(q, qd).detach().cpu().numpy()

_q6 = _states6[:, :6].numpy()
_rmin6 = np.minimum.reduce([np.linalg.norm(_q6[:, 0:2]-_q6[:, 2:4], axis=1),
                           np.linalg.norm(_q6[:, 2:4]-_q6[:, 4:6], axis=1),
                           np.linalg.norm(_q6[:, 4:6]-_q6[:, 0:2], axis=1)])

_a_lj    = _analytic_accel6(_p_lj,   _states6)
_a_at1   = _analytic_accel6(_p_at1,  _states6)
_a_full_amp  = _analytic_accel6(_p_amp,  _states6)
_a_full_phys = _analytic_accel6(_p_phys, _states6)
_pred_lj   = _model_accel6(_model_lj,   _states6)
_pred_amp  = _model_accel6(_model_amp,  _states6)
_pred_phys = _model_accel6(_model_phys, _states6)

def _rn(a): return np.linalg.norm(a, axis=1)
_signal_phys = _NU_PHYS * _rn(_a_at1)
_signal_amp  = _NU_AMP  * _rn(_a_at1)
_floor       = _rn(_pred_lj - _a_lj)
_res_lj_on_amp   = _rn(_pred_lj  - _a_full_amp)
_res_full_amp    = _rn(_pred_amp - _a_full_amp)
_res_lj_on_phys  = _rn(_pred_lj  - _a_full_phys)
_res_full_phys   = _rn(_pred_phys - _a_full_phys)

def _med6(x, y, b):
    m, e, _ = binned_statistic(x, y, statistic="median", bins=b)
    c = 0.5*(e[:-1]+e[1:]); ok = np.isfinite(m)
    return c[ok], m[ok]

_bins6 = np.linspace(np.percentile(_rmin6, 1), np.percentile(_rmin6, 99), 20)
fig, ax = plt.subplots(1, 2, figsize=(_PAGE_W, _PAGE_W*0.40))

a0 = ax[0]
a0.plot(*_med6(_rmin6, _floor, _bins6),       color=_C_FLOOR, lw=1.8, label="pairwise noise floor")
a0.plot(*_med6(_rmin6, _signal_amp, _bins6),  color=_C_AMP,  lw=1.8, label=r"AT signal, $\tilde\nu=1.0$")
a0.plot(*_med6(_rmin6, _signal_phys, _bins6), color=_C_PHYS, lw=1.8, ls="--",
        label=r"AT signal, $\tilde\nu=0.1$")
a0.set_yscale("log"); a0.set_xlabel(r"$r_{\min}$"); a0.set_ylabel("acceleration magnitude")
a0.set_title("(a) three-body signal vs. model floor"); a0.grid(alpha=0.3, which="both")
a0.legend(fontsize=7)

a1 = ax[1]
a1.plot(*_med6(_rmin6, _res_lj_on_amp, _bins6),  color=_C_FLOOR, lw=1.8, label=r"pairwise, $\tilde\nu=1.0$")
a1.plot(*_med6(_rmin6, _res_full_amp, _bins6),   color=_C_AMP,  lw=1.8, label=r"full model, $\tilde\nu=1.0$")
a1.plot(*_med6(_rmin6, _res_lj_on_phys, _bins6), color=_C_FLOOR, lw=1.4, ls="--",
        label=r"pairwise, $\tilde\nu=0.1$")
a1.plot(*_med6(_rmin6, _res_full_phys, _bins6),  color=_C_PHYS, lw=1.4, ls="--",
        label=r"full model, $\tilde\nu=0.1$")
a1.set_yscale("log"); a1.set_xlabel(r"$r_{\min}$")
a1.set_ylabel(r"$\|a_\mathrm{pred}-a_\mathrm{true}\|$")
a1.set_title("(b) residual vs. true composite dynamics"); a1.grid(alpha=0.3, which="both")
a1.legend(fontsize=7)

# (no fig.suptitle)
fig.tight_layout()
fig.savefig(f"{_OUT_DIR}/fig6_identifiability.pdf", bbox_inches="tight")
print("fig6 done")

## Fig. 7: Newtonian gravity control

In [ ]:
# =============================================================================
# Control: purely pairwise Newtonian gravity, same (r, cos-theta) representation.
# Gravity has NO three-body term -- correct recovery here is a second, structurally
# different confirmation (1/r radial law, not LJ's 12-6) that the invariant
# featurization does not inject spurious angular/three-body structure. We also
# include the distances-only (3-feature) ablation trained on identical data.
#
# NOTE: the shared CombinedNetwork.forward hard-codes 6 features, so it cannot run
# a 3-feature checkpoint. We therefore detect each model's input dim, build the
# matching features ourselves, and read V = mlp(features) / accel = -grad V/m
# directly (valid here: the mass matrix is the identity, so a = -grad V / m).
# =============================================================================
import sys, os
sys.path.insert(0, str(LNN_DIR))
import numpy as np, torch
import matplotlib as mpl, matplotlib.pyplot as plt
from physical_systems import newt_three_body
from fixedkinetic_LNN import LNN
torch.serialization.add_safe_globals([LNN])
from release_io import load_model_path as load_model

PROJ = str(WEIGHTS)
OUT_DIR = str(FIGURES)
DEV, DT = torch.device("cpu"), torch.float32
try:
    set_pub_style()
except NameError:
    mpl.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm",
                          "font.size": 9, "savefig.dpi": 300, "pdf.fonttype": 42})
_COL_W  = PRE_COL_W  if "PRE_COL_W"  in dir() else 3.375
_PAGE_W = PRE_PAGE_W if "PRE_PAGE_W" in dir() else 7.0

MODELS = [
    ("newtonian_threebody_r01_5",      "full $(r,\\cos\\theta)$", "#0072B2"),
    ("newtonian_threebody_r01_5_3DOF", "distances only",          "#D55E00"),
]
R_MIN, R_MAX = 0.1, 5.0
p_newt = newt_three_body()            # G=1, m=1, scale=1; V=-sum m^2/r, no E0/eps_LJ

def build_features(q, in_dim):
    """Replicates CombinedNetwork.forward's featurization for 3- or 6-input MLPs."""
    q = q.view(-1, 3, 2)
    r12 = torch.norm(q[:,0]-q[:,1], dim=1, keepdim=True)
    r23 = torch.norm(q[:,1]-q[:,2], dim=1, keepdim=True)
    r31 = torch.norm(q[:,2]-q[:,0], dim=1, keepdim=True)
    if in_dim == 3:
        return torch.cat([r12, r23, r31], dim=1)
    v12, v13, v23 = q[:,1]-q[:,0], q[:,2]-q[:,0], q[:,2]-q[:,1]
    c1 = (v12*v13).sum(1, keepdim=True) / (r12*r31 + 1e-6)
    c2 = (-v12*v23).sum(1, keepdim=True) / (r12*r23 + 1e-6)
    c3 = ((-v13)*(-v23)).sum(1, keepdim=True) / (r31*r23 + 1e-6)
    return torch.cat([r12, r23, r31, c1, c2, c3], dim=1)

def learned_V(model, Q):
    in_dim = model.network.mlp[0].in_features
    with torch.no_grad():
        return model.network.mlp(build_features(Q, in_dim)).squeeze(-1).numpy()

def learned_A(model, Q):
    in_dim = model.network.mlp[0].in_features
    q = Q.clone().requires_grad_(True)
    V = model.network.mlp(build_features(q, in_dim)).sum()
    g = torch.autograd.grad(V, q)[0]
    return (-g / p_newt.m).detach().numpy()

def sample_valid(n, seed=0, box=2.0):
    rng = np.random.default_rng(seed); out = []
    while len(out) < n:
        pts = rng.uniform(-box, box, size=(3,2))
        d = [np.linalg.norm(pts[a]-pts[b]) for a,b in [(0,1),(1,2),(2,0)]]
        if min(d) < R_MIN or max(d) > R_MAX: continue
        out.append((pts - pts.mean(0)).reshape(-1).astype(np.float32))
    return np.stack(out)

Q  = torch.as_tensor(sample_valid(800), dtype=DT)
Z  = torch.zeros_like(Q)
with torch.no_grad():
    Vt = p_newt.potential(Q, Z).numpy()
_q = Q.clone().requires_grad_(True); _z = Z.clone().requires_grad_(True)
At = p_newt.solve_acceleration(_q, _z).detach().numpy()
At_norm = np.linalg.norm(At, axis=1)

results = {}
print("=== Newtonian gravity control: relative metrics ===")
for d, label, color in MODELS:
    path = f"{PROJ}/{d}/model.pt"
    if not os.path.exists(os.path.dirname(path)):
        print(f"  {label}: no model.pt at {d}, skipping"); continue
    model = load_model(path, DEV, DT).eval()
    Vp = learned_V(model, Q)
    Ap = learned_A(model, Q)
    offset = np.median(Vp - Vt)                       # gauge-fix additive constant once
    Vp -= offset
    rel_V = np.mean(np.abs(Vp - Vt) / np.abs(Vt))
    rel_A = np.mean(np.linalg.norm(Ap - At, axis=1) / At_norm)
    r2    = 1 - np.sum((Vp - Vt)**2) / np.sum((Vt - Vt.mean())**2)
    results[label] = dict(Vp=Vp, rel_V=rel_V, rel_A=rel_A, r2=r2, color=color)
    print(f"  {label:22s}: rel. V error = {100*rel_V:5.2f}%   "
          f"rel. accel error = {100*rel_A:5.2f}%   R^2(V) = {r2:.4f}")

# --- parity figure: learned vs analytic potential ---
fig, ax = plt.subplots(1, 1, figsize=(_COL_W, 3.0))
lo, hi = Vt.min(), Vt.max()
ax.plot([lo, hi], [lo, hi], "k--", lw=0.8, zorder=1, label="identity")
for label, r in results.items():
    ax.scatter(Vt, r["Vp"], s=6, alpha=0.4, color=r["color"], zorder=2,
               label=f"{label} ($R^2$={r['r2']:.4f})")
ax.set_xlabel(r"analytic $V_{\rm Newt}$"); ax.set_ylabel(r"learned $V$ (gauge-fixed)")
ax.legend(fontsize=6, loc="upper left")
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/fig7_newtonian_control.pdf", bbox_inches="tight")
print("fig7 done")

## Fig. 6: pure-LJ control

`pure_AT_final_figures.ipynb` produced this figure by hand-editing its setup cell (the commented
"CHANGE IN BETWEEN MODELS" block: LJ model, ν = 0, r_min = 0.9, r_max = 3.0). The next cell does the same switch
explicitly, then runs the original figure cell.

In [ ]:
PROJ = str(WEIGHTS)
OUT_DIR = str(FIGURES)
RUN_DIR = f"{PROJ}/argonlike-liquid_pure_LJ"
NU, R_MIN, R_MAX = 0.0, 0.9, 3.0
model = load_model(f"{RUN_DIR}/model.pt", DEVICE, DTYPE); model.eval()
particle = load_particle(initialize_particle_system("three_body"), f"{RUN_DIR}/particle.json")
particle.E0 = NU
LEARN_V_OFFSET = 0.0
LEARN_V_OFFSET = _calibrate_offset()
print("switched to", RUN_DIR, "| E0 =", particle.E0, "eps_LJ =", particle.eps_LJ, "| gauge offset =", LEARN_V_OFFSET)

In [ ]:
# ===========================================================================
# FIGURE 2 : radial dependence, 6 configurations, potential AND acceleration
# ===========================================================================
def _center(pts):
    pts = pts - pts.mean(axis=0)
    return pts.reshape(-1).astype(np.float32)

def build_equilateral(r): return _center(np.array([[0,0],[r,0],[0.5*r, np.sqrt(3)/2*r]]))
def build_right(r):       return _center(np.array([[0,0],[r,0],[0,r]]))
def build_line(r):        return _center(np.array([[-r,0],[0,0],[r,0]]))
def build_far(r):         return _center(np.array([[0,0],[r,0],[0,3*r]]))
def build_isosceles(r, sr=1.25):
    s = sr*r
    h = np.sqrt(s**2 - (r/2)**2)
    return _center(np.array([[0,0],[r,0],[r/2,h]]))
def build_scalene(r):     return _center(np.array([[0,0],[1.2*r,0],[0.35*r,1.05*r]]))

CONFIGS = {
    "Equilateral": build_equilateral,
    "Right": build_right,
    "Line": build_line,
    "Far particle": build_far,
    "Isosceles": build_isosceles,
    "Scalene": build_scalene,
}

def fig2_radial(rvals=None, save=True):
    if rvals is None:
        rvals = np.linspace(R_MIN, 1.6, 90)

    fig, axes = plt.subplots(4, 3, figsize=(PRE_PAGE_W, PRE_PAGE_W*1.05))

    for c, (name, builder) in enumerate(CONFIGS.items()):
        col = c % 3
        block = c // 3
        rowV = 0 if block == 0 else 1
        rowA = 2 if block == 0 else 3

        Vt, Vp, At, Ap = [], [], [], []

        for r in rvals:
            q6 = builder(r)
            Vt.append(true_potential(q6))
            Vp.append(learned_potential(q6))
            At.append(np.linalg.norm(true_accel(q6)))
            Ap.append(np.linalg.norm(learned_accel(q6)))

        Vt = np.asarray(Vt)
        Vp = np.asarray(Vp)
        At = np.asarray(At)
        Ap = np.asarray(Ap)

        # Center potentials (energy offset is arbitrary)
        Vt -= Vt.mean()
        Vp -= Vp.mean()

        # Reference power laws (also centered for comparison)
        refV = Vt[0] * (rvals / rvals[0])**(-9)
        refV -= refV.mean()
        refA = At[0] * (rvals / rvals[0])**(-10)

        axv = axes[rowV, col]
        axv.plot(rvals, Vt, color=C_TRUE, label="Analytic")
        axv.plot(rvals, Vp, color=C_LEARN, ls="--", label="Learned")
        #axv.plot(rvals, refV, color="0.6", lw=0.8, ls=":", label=r"$r^{-9}$")
        axv.set_title(name, fontsize=9)
        axv.grid(alpha=0.3)
        if col == 0:
            axv.set_ylabel(r"$V-\langle V\rangle$")

        axa = axes[rowA, col]
        axa.plot(rvals, At, color=C_TRUE)
        axa.plot(rvals, Ap, color=C_LEARN, ls="--")
        #axa.plot(rvals, refA, color="0.6", lw=0.8, ls=":")
        axa.grid(alpha=0.3)
        axa.set_xlabel(r"$r$")
        if col == 0:
            axa.set_ylabel(r"$|\mathbf{a}|$")

    axes[0, 0].legend(loc="lower left", fontsize=6)

    fig.tight_layout()

    if save:
        fig.savefig(f"{OUT_DIR}/fig2_radial_raw_lj.pdf", bbox_inches="tight")

    print("fig2 done")
    return fig


# ===========================================================================

fig2_radial()

## Check: every file the `.tex` includes is here

In [ ]:
import os
TEX_FILES = ["at_familiarity_schematic.pdf", "sampling_ablation.pdf", "fig1_field_ticked.pdf", "fig2_radial_raw.pdf",
             "fig2_radial_raw_lj.pdf", "fig7_newtonian_control.pdf", "fig3_angular_raw.pdf", "fig4a_curved.pdf",
             "fig4b_ensemble.pdf", "fig5_special.pdf", "fig6_identifiability.pdf"]
FIG_DIR = str(FIGURES)
for f in TEX_FILES:
    p = f"{FIG_DIR}/{f}"
    print(f"{'OK     ' if os.path.exists(p) else 'MISSING'} {f}")
print("\nNumbers printed above should equal the originals: Fig. 4 rel V 1.30% / rel F 1.08%; Fig. 3 sweep 0.088/0.008/0.135 vs "
      "1.004/1.249/7.472; Fig. 8 0.81% / 1.06%; Newtonian 2.6% / 2.5%, R^2 0.9999 / 0.9997.")